# Quick Start

A complete walkthrough of a core PyTorch workflow: loading data, building a model, training it, and evaluating its performance.

**What you'll learn:**
- Loading and preprocessing image data with `Dataset` and `DataLoader`
- Defining a neural network using `nn.Module`
- Setting up a loss function with `nn.CrossEntropyLoss` and optimizer with `torch.optim.SGD`
- Running a full training and evaluation loop
- Saving and Loading a model

## Working with Data

PyTorch provides two core abstractions for data pipelines:

- **`Dataset`** — stores samples and their corresponding labels
- **`DataLoader`** — wraps a `Dataset` in an iterable with built-in support for batching, shuffling, and parallel loading

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

PyTorch offers domain-specific libraries — TorchText, TorchVision, and TorchAudio — each bundling ready-to-use datasets. Here we use **FashionMNIST** from `torchvision.datasets`: 70,000 grayscale clothing images across 10 categories.

Every TorchVision `Dataset` accepts two optional arguments: `transform` to preprocess input images, and `target_transform` to preprocess labels.

In [17]:
# Download training data from open datasets
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose(
        [
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ]
    ),
)

# Download test data from open datasets
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform= v2.Compose(
        [
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ]
    ),
)


We pass the `Dataset` to a `DataLoader`, which wraps it in an iterable that handles automatic batching, sampling, shuffling, and multiprocess loading. A batch size of 64 means each iteration yields 64 image–label pairs.

In [18]:
batch_size = 64

# Create DataLoaders
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print("shape of X: ", X.shape)
    print("shape of y: ", y.shape)
    break

shape of X:  torch.Size([64, 1, 28, 28])
shape of y:  torch.Size([64])


## Creating Models

A PyTorch neural network is defined as a class that inherits from `nn.Module`. Layers are declared in `__init__`, and the data flow through the network is defined in `forward`.

We also move the model to the best available hardware accelerator (CUDA, MPS, MTIA, or XPU). If none is found, training falls back to CPU.

In [19]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available else "cpu"
print(f"Using device: ", device)

Using device:  cuda


In [20]:
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.Flatten = nn.Flatten()
        self.Linear_Stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )
    
    def forward(self, x):
        x = self.Flatten(x)
        x = self.Linear_Stack(x)
        return x

model = NeuralNet().to(device)
print(model)

NeuralNet(
  (Flatten): Flatten(start_dim=1, end_dim=-1)
  (Linear_Stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=1024, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): ReLU()
    (6): Linear(in_features=512, out_features=256, bias=True)
    (7): ReLU()
    (8): Linear(in_features=256, out_features=10, bias=True)
  )
)


## Optimizing Model Parameters

Training requires two components:

- **Loss function** — measures how far the model's predictions are from the ground truth. `CrossEntropyLoss` is the standard choice for multi-class classification.
- **Optimizer** — updates the model's weights to minimize the loss. We use `SGD` with momentum (helps escape local minima) and weight decay (L2 regularization to reduce overfitting).

In [25]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

In each training step, the model makes predictions on a batch, computes the loss against the true labels, backpropagates the error, and updates the weights.

In [22]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        # compute loss
        label = model(X)
        loss = loss_fn(label, y)
        # loss back propagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f},  [{current:>5d}/{size:>5d}]")

We also evaluate on the held-out test set after each epoch. The model is switched to `eval()` mode — this disables dropout and stabilizes batch normalization. Gradients are not computed here.

In [23]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            label = model(X)
            loss = loss_fn(label, y)
            test_loss += loss.item()
            correct += (label.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test: Avg Loss {test_loss:>7f}, Acc {correct*100:>.1f}%")

We train for `num_epochs` passes over the full dataset. Each epoch runs a full training sweep followed by evaluation on the test set. As training progresses, **loss should decrease and accuracy should increase**.

In [26]:
num_epochs = 5
for t in range(num_epochs):
    print(f"Epoch {t+1}:-----------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1:-----------------------------
loss: 2.307971,  [   64/60000]
loss: 2.301012,  [ 6464/60000]
loss: 2.286276,  [12864/60000]
loss: 2.277073,  [19264/60000]
loss: 2.266748,  [25664/60000]
loss: 2.226243,  [32064/60000]
loss: 2.198500,  [38464/60000]
loss: 2.039589,  [44864/60000]
loss: 1.803667,  [51264/60000]
loss: 1.487077,  [57664/60000]
Test: Avg Loss 1.387141, Acc 45.2%
Epoch 2:-----------------------------
loss: 1.453390,  [   64/60000]
loss: 1.295598,  [ 6464/60000]
loss: 1.042351,  [12864/60000]
loss: 1.076131,  [19264/60000]
loss: 0.909943,  [25664/60000]
loss: 0.864298,  [32064/60000]
loss: 0.927673,  [38464/60000]
loss: 0.866978,  [44864/60000]
loss: 0.809923,  [51264/60000]
loss: 0.784966,  [57664/60000]
Test: Avg Loss 0.772043, Acc 70.6%
Epoch 3:-----------------------------
loss: 0.723823,  [   64/60000]
loss: 0.847738,  [ 6464/60000]
loss: 0.581372,  [12864/60000]
loss: 0.765986,  [19264/60000]
loss: 0.676366,  [25664/60000]
loss: 0.556180,  [32064/60000]
loss: 0.72

## Saving and Loading Models

A common way to save a model is to serialize its internal state dictionary

In [33]:
torch.save(model.state_dict(), "neuralnet.pth")
print("Model 'neuralnet' has been successfully saved.")

Model 'neuralnet' has been successfully saved.


The process for loading a model includes re-creating the model structure and loading the state dictionary into it.

In [34]:
model = NeuralNet().to(device)
model.load_state_dict(torch.load("neuralnet.pth", weights_only=False))
print("Model has been successfully loaded.")

Model has been successfully loaded.


This model can be used for predicting labels.

In [35]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    label = model(x)
    pred, actual = classes[label[0].argmax(0)], classes[y]
    print(f"Prediction: {pred}, Actual: {actual}")

Prediction: Ankle boot, Actual: Ankle boot
